# Shopping Data Analysis - Assignment
Python & Pandas basics - EDA and cleaning

---

## 1. Importing Libraries

In [ ]:
# first lets import everything we need
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# make plots look decent
plt.style.use('ggplot')  
print('libraries imported successfully')

## 2. Loading the Data
Using the combined_dataset.csv file that has all product information.

In [ ]:
# load dataset
df = pd.read_csv('../data/combined_dataset.csv')

# quick look at first few rows
print('First 5 rows:')
display(df.head())

# and last few
print('\nLast 5 rows:')
display(df.tail())

# check size
print(f'\nShape: {df.shape}')
print(f'Total columns: {len(df.columns)}')

## 3. Understanding the Data
Lets check what kind of data we are dealing with.

In [ ]:
# column names
print('Columns in dataset:')
for i, col in enumerate(df.columns):
    print(f'  {i+1}. {col}')

print('\n---')

# data types and info
df.info()

print('\n---')
print('Statistical summary:')
display(df.describe(include='all'))

### Observations so far:
- 1000 rows and 24 columns - decent sized dataset
- final_price is stored as string (object type) because of the ₹ symbol and commas
- some columns have missing values that we need to handle
- most of the data is related to product listings (prices, ratings, categories)

## 4. Checking Missing Values & Duplicates

In [ ]:
# check whats missing
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0])

print(f'\nTotal duplicate rows: {df.duplicated().sum()}')

## 5. Data Cleaning
Time to clean this up! We need to:
- fix the price column formatting
- handle missing values
- remove any duplicates

In [ ]:
# make a copy so we dont mess with original
df_clean = df.copy()

# remove duplicate rows if any
df_clean = df_clean.drop_duplicates()
print(f'After removing duplicates: {df_clean.shape[0]} rows')

# cleaning final_price - it has ₹, quotes and commas
# need to strip all that and convert to number
df_clean['final_price'] = df_clean['final_price'].astype(str)
df_clean['final_price'] = df_clean['final_price'].str.replace('₹', '')
df_clean['final_price'] = df_clean['final_price'].str.replace(',', '')
df_clean['final_price'] = df_clean['final_price'].str.replace('"', '')
df_clean['final_price'] = pd.to_numeric(df_clean['final_price'], errors='coerce')

# renaming for easier use
df_clean.rename(columns={'final_price': 'price'}, inplace=True)

# also make sure initial_price is numeric
df_clean['initial_price'] = pd.to_numeric(df_clean['initial_price'], errors='coerce')

print('Price columns cleaned successfully')
print(f'Price range: ₹{df_clean["price"].min():.0f} - ₹{df_clean["price"].max():.0f}')

In [ ]:
# now handling missing values
# for numerical columns - fill with median (safer than mean when there are outliers)
num_cols = ['rating', 'ratings_count', 'initial_price', 'price']
for col in num_cols:
    if df_clean[col].isnull().sum() > 0:
        med = df_clean[col].median()
        df_clean[col].fillna(med, inplace=True)
        print(f'Filled missing in {col} with median: {med}')

# for text columns - fill with most common value
cat_cols = ['brand', 'currency', 'category', 'seller_name', 'delivery_options']
for col in cat_cols:
    if col in df_clean.columns and df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f'Filled missing in {col} with mode: {mode_val}')

# filling some specific columns with placeholders
df_clean['what_customers_said'].fillna('No feedback', inplace=True)
df_clean['variations'].fillna('None', inplace=True)

print('\nMissing value handling complete!')
print(f'Remaining missing values: {df_clean.isnull().sum().sum()}')

In [ ]:
# extract brand name from title column
df_clean['brand'] = df_clean['title'].str.strip()

# standardize category names to lowercase
df_clean['category'] = df_clean['category'].str.lower().str.strip()

print('Text standardization done')

## 6. Basic Pandas Operations
Lets try some filtering, selecting, sorting etc.

In [ ]:
# select specific columns
print('Selected columns (brand, price):')
display(df_clean[['brand', 'price']].head())

# filter - products above ₹3000
expensive = df_clean[df_clean['price'] > 3000]
print(f'\nProducts with price > ₹3000: {len(expensive)} products')
display(expensive[['brand', 'price', 'rating']].head(8))

# sort by price - most expensive first
print('\nTop 5 most expensive products:')
display(df_clean.sort_values('price', ascending=False)[['brand', 'price', 'rating']].head(5))

# group by brand - average price and rating
print('\nBrand-wise averages:')
brand_stats = df_clean.groupby('brand')[['price', 'rating']].mean().round(2)
display(brand_stats.head(10))

# whats the count per brand?
print('\nTop 10 brands by product count:')
print(df_clean['brand'].value_counts().head(10))

## 7. Feature Engineering
Creating new columns to get more insights from the data.

In [ ]:
# price difference = how much discount you actually get
df_clean['price_diff'] = df_clean['initial_price'] - df_clean['price']

# discount percentage
df_clean['discount_pct'] = (df_clean['price_diff'] / df_clean['initial_price'] * 100).round(2)
df_clean['discount_pct'].fillna(0, inplace=True)

# popularity score = rating * number of ratings
# this helps find products that are both highly rated AND have many reviews
df_clean['popularity'] = df_clean['rating'] * df_clean['ratings_count']

# creating quantity column
# since we dont have actual sales quantity, using ratings_count as proxy
# (more ratings usually means more people bought it)
df_clean['quantity'] = df_clean['ratings_count']

# total amount = price * quantity (derived column as per assignment)
df_clean['total_amount'] = df_clean['price'] * df_clean['quantity']

print('New features created:')
print(df_clean[['brand', 'price', 'price_diff', 'discount_pct', 'quantity', 'total_amount', 'popularity']].head())

## 8. Univariate Analysis
Looking at one variable at a time.

In [ ]:
# make dir for saving plots
import os
if not os.path.exists('../output'):
    os.makedirs('../output')

# 1. price distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_clean['price'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Product Prices')
plt.xlabel('Price (₹)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../output/price_distribution.png', dpi=100)
plt.show()

# 2. top brands bar chart
top15 = df_clean['brand'].value_counts().head(15).index
plt.figure(figsize=(12, 6))
sns.countplot(data=df_clean[df_clean['brand'].isin(top15)], y='brand', order=top15, palette='viridis')
plt.title('Top 15 Brands by Product Count')
plt.xlabel('Number of Products')
plt.tight_layout()
plt.savefig('../output/top_brands.png', dpi=100)
plt.show()

# 3. rating boxplot
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_clean['rating'], color='lightcoral')
plt.title('Customer Ratings Distribution')
plt.tight_layout()
plt.savefig('../output/rating_boxplot.png', dpi=100)
plt.show()

print('All univariate plots saved!')

## 9. Bivariate Analysis
Now lets see how variables relate to each other.

In [ ]:
# price vs rating scatter
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_clean, x='price', y='rating', alpha=0.4, color='teal')
plt.title('Price vs Rating')
plt.xlabel('Price (₹)')
plt.ylabel('Rating')
plt.tight_layout()
plt.savefig('../output/price_vs_rating.png', dpi=100)
plt.show()

# brand price comparison (top 10)
top10 = df_clean['brand'].value_counts().head(10).index
plt.figure(figsize=(12, 5))
sns.barplot(data=df_clean[df_clean['brand'].isin(top10)], x='brand', y='price', palette='Set2')
plt.title('Average Price by Brand (Top 10)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../output/brand_prices.png', dpi=100)
plt.show()

# correlation heatmap
num_features = ['initial_price', 'price', 'price_diff', 'discount_pct', 'rating', 'ratings_count', 'popularity', 'total_amount']
plt.figure(figsize=(10, 8))
sns.heatmap(df_clean[num_features].corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.savefig('../output/correlation_heatmap.png', dpi=100)
plt.show()

print('Bivariate plots saved!')

## 10. Category Analysis
Breaking it down by product category.

In [ ]:
# group by category
cat_stats = df_clean.groupby('category').agg({
    'price': ['mean', 'min', 'max'],
    'rating': 'mean',
    'total_amount': 'sum',
    'product_id': 'count'
}).round(2)

cat_stats.columns = ['avg_price', 'min_price', 'max_price', 'avg_rating', 'total_revenue', 'product_count']
print('Category-wise breakdown:')
display(cat_stats.sort_values('product_count', ascending=False))

## 11. Saving Cleaned Data
Exporting the cleaned dataset with all the new features we created.

In [ ]:
# save to csv
df_clean.to_csv('../output/cleaned_dataset.csv', index=False)
print('Cleaned dataset saved!')
print(f'Final shape: {df_clean.shape}')
print(f'Columns: {list(df_clean.columns)}')

## 12. Key Takeaways

**What we found:**
- Product prices vary widely across categories. Some categories have mostly budget items while others have premium products.
- Most products have ratings between 3.5 and 4.5 - very few products are rated below 3.
- Some brands clearly dominate certain categories.
- The quantity/total_amount calculation gives us an estimate of which products are generating the most value.

**Business stuff:**
- Brands with high popularity scores should be prioritized for inventory and marketing.
- Low rated products (below 3.5) need investigation - might be quality issues.
- Categories with high avg prices might need more detailed product descriptions to justify cost.